# FactLedger extractor

`load(path) -> documents, units` over raw files and nothing else. The extractor sniffs the
format from the bytes and writes document and unit JSON in the shapes of SCHEMA.md; the
rules it follows are in BUILD.md. Built one block at a time. Inputs: the public raw dataset
and the private papers dataset, both attached to this notebook.


In [ ]:
# Block 1: inputs and integrity.
# Mount both datasets, count files per folder, and check every file's sha256 against the
# folder manifest. The manifests are used here only to prove the Kaggle copies are the bytes
# that were uploaded; the extractor itself never reads them.
import hashlib
import json
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

def mount(slug):
    """Kaggle mounts inputs at /kaggle/input/<slug> or, in newer sessions,
    /kaggle/input/datasets/<owner>/<slug>. Take whichever exists."""
    for candidate in (Path("/kaggle/input") / slug, Path("/kaggle/input/datasets/jhffmn") / slug):
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError(slug)


RAW = mount("it494-narrative-corpora-raw")
PAPERS = mount("it494-reference-papers")


def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def check(folder, rows_key):
    manifest = json.loads((folder / "manifest.json").read_text(encoding="utf-8"))
    rows = manifest[rows_key]
    on_disk = {p.name for p in folder.iterdir() if p.name not in ("manifest.json", "LICENSE")}
    listed = {r["file"] for r in rows}
    # Kaggle inputs are a network filesystem: one file at a time, 19,206 files take tens of
    # minutes; 32 concurrent reads take about a minute.
    with ThreadPoolExecutor(max_workers=32) as pool:
        digests = list(pool.map(sha256, [folder / r["file"] for r in rows]))
    bad = [r["file"] for r, d in zip(rows, digests) if d != r["sha256"]]
    print(f"{folder.name:<28} files {len(on_disk):>6}  listed {len(listed):>6}"
          f"  mismatched {len(bad)}  unlisted {len(on_disk - listed)}  missing {len(listed - on_disk)}")
    return bad


# The three literature manifests keep their original "works" key; the unpacked folders
# and the papers use "files".
for name, key in [("oz", "works"), ("holmes", "works"), ("greek", "works"),
                  ("graphrag-bench", "files"), ("longmemeval", "files")]:
    check(RAW / name, key)
check(PAPERS, "files")


In [ ]:
# Block 2: file type, then raw text.
#
# Two steps, by bytes only. Nothing here decides what the text is about; that is the model's
# job later.
#   1. file_kind(data): look at the first bytes and name the container: pdf, json, or text.
#   2. to_text(path): turn the container into one string, the document text.
#        pdf  -> the text layer, page by page (PyMuPDF, the one dependency)
#        json -> if it holds chat turns, one "role: content" block per turn under a header
#                of the session id and dates. We also keep where each turn starts and ends
#                in that string, so a chat can be cut into units without a model.
#        text -> the bytes decoded as UTF-8, unchanged
import json

try:
    import pymupdf
except ImportError:
    import subprocess
    import sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pymupdf"], check=True)
    import pymupdf


def file_kind(data):
    if data.startswith(b"%PDF-"):
        return "pdf"
    if data.lstrip()[:1] in (b"{", b"["):
        return "json"
    return "text"


def pdf_text(data):
    doc = pymupdf.open(stream=data, filetype="pdf")
    return "\n".join(page.get_text() for page in doc)


def chat_turns(obj):
    """The list of {role, content} turns inside a chat JSON, wherever it sits; None if absent."""
    if isinstance(obj, list) and obj and all(isinstance(t, dict) and "role" in t and "content" in t for t in obj):
        return obj
    if isinstance(obj, dict):
        for value in obj.values():
            found = chat_turns(value)
            if found:
                return found
    return None


def chat_text(obj, turns):
    """Header lines, a blank line, then 'role: content' per turn. Returns the text and the
    (start, end) of each turn inside it."""
    header = [f"session_id: {obj['session_id']}"] if "session_id" in obj else []
    header += [f"date: {d}" for d in obj.get("dates", [])]
    text = "\n".join(header) + "\n\n"
    spans = []
    for turn in turns:
        start = len(text)
        text += f"{turn['role']}: {turn['content']}\n\n"
        spans.append((start, len(text)))
    return text, spans


def to_text(path):
    data = path.read_bytes()
    doc = {"path": str(path), "sha256": hashlib.sha256(data).hexdigest(),
           "kind": file_kind(data), "text": "", "turns": None, "dates": []}
    if doc["kind"] == "pdf":
        doc["text"] = pdf_text(data)
    elif doc["kind"] == "json":
        obj = json.loads(data)
        turns = chat_turns(obj)
        if turns is None:
            doc["kind"], doc["text"] = "text", data.decode("utf-8", errors="replace")
        else:
            doc["kind"] = "chat"
            doc["text"], doc["turns"] = chat_text(obj, turns)
            doc["dates"] = list(obj.get("dates", []))
    else:
        doc["text"] = data.decode("utf-8", errors="replace")
    return doc


# One of each, to see the shape.
for path in [RAW / "oz" / "01_55.txt", RAW / "graphrag-bench" / "Novel-30752.txt",
             RAW / "longmemeval" / "sharegpt_yywfIrx_0.json", RAW / "longmemeval" / "001cefa7_2.json",
             PAPERS / "edge2024-graphrag.pdf"]:
    d = to_text(path)
    turns = len(d["turns"]) if d["turns"] else "-"
    print(f"{path.name:<26} {d['kind']:<5} {len(d['text']):>8,} chars  turns {turns:>3}  dates {d['dates']}")
    print("    " + repr(d["text"][:70]))


In [ ]:
# Block 3: the model call, and the check that a quoted line really exists in the text.
import json
import re
import time

import requests
from kaggle_secrets import UserSecretsClient

MODEL = "gpt-5.6-luna"
RETRY = "gpt-5.6-terra"
PRICE = {"gpt-5.6-luna": (0.20, 1.20), "gpt-5.6-terra": (2.00, 12.00)}   # $ per M tokens in, out
SPEND_STOP = 8.00                                                        # dollars; the run halts past this
KEY = UserSecretsClient().get_secret("OPENAI_API_KEY")
calls = []


def spend():
    return sum(c["cost"] for c in calls)


def generate(prompt, model=MODEL, effort="low"):
    """One JSON-mode call; the reply parsed, the cost logged. The API rejects temperature."""
    if spend() >= SPEND_STOP:
        raise RuntimeError(f"spending stop: ${spend():.2f}")
    t0 = time.time()
    r = requests.post("https://api.openai.com/v1/chat/completions",
                      headers={"Authorization": f"Bearer {KEY}"}, timeout=300,
                      json={"model": model, "reasoning_effort": effort,
                            "response_format": {"type": "json_object"},
                            "messages": [{"role": "user", "content": prompt}]})
    if r.status_code != 200:
        raise RuntimeError(f"OpenAI {r.status_code}: {r.text}")
    body = r.json()
    u, (p_in, p_out) = body["usage"], PRICE[model]
    calls.append({"model": body["model"], "in": u["prompt_tokens"], "out": u["completion_tokens"],
                  "seconds": round(time.time() - t0, 1),
                  "cost": (u["prompt_tokens"] * p_in + u["completion_tokens"] * p_out) / 1e6})
    return json.loads(body["choices"][0]["message"]["content"])


def locate(text, line, start=0):
    """Character position where the quoted line occurs as a whole line, at or after start,
    spacing forgiven. None when it is not there."""
    words = (line or "").split()
    if not words:
        return None
    rx = r"(?m)^[ \t]*" + r"\s+".join(re.escape(w) for w in words) + r"[ \t]*\r?$"
    m = re.compile(rx).search(text, start)
    return None if m is None else m.start()


print(f"model {MODEL}, retry {RETRY}, spend stop ${SPEND_STOP:.2f}, key {'present' if KEY else 'MISSING'}")


In [ ]:
# Block 5: the split call, as a function. The text goes to the model in slices of about
# 50,000 tokens, cut at line breaks; every slice gets the same question; the answers are
# joined into one reply.
SLICE = 200_000      # characters per call
CAP_WORDS = 4000

PROMPT = """Below is part %d of %d of one document, as raw text. Answer with JSON only. Every string you return must be copied from the text exactly, character for character, never paraphrased or corrected, because a program will search the text for it.

{
  "source_class": one of "canonical" (a published literary or classic work), "published" (a paper, article, or report), "authored" (a person's own material: notes, email, letters, drafts); null unless this is part 1,
  "title": the title as written, or null,
  "author": the author's name as written, or null,
  "date": {"quote": the complete line containing the date the work was written, published, or sent, "iso": "YYYY" or "YYYY-MM" or "YYYY-MM-DD"} or null. Not a transcription or ebook release date,
  "body_start": the complete first line of the work itself, or null if the work does not begin in this part. Publisher notices, a contents list, and transcriber's or translator's notes are not part of the work; an author's own preface or introduction is,
  "end_matter_start": the complete first line of any end matter that follows the work (license, index, notes, advertisements), or null if this part has no such line,
  "toc_count": the number of pieces a contents list gives, or null,
  "pieces": [{"marker": the complete heading line that begins a piece in this part, exactly as it appears where the piece begins, not as a contents list writes it, "title": a short title for the piece, such as "Chapter 1: The Cyclone" or "Abstract" or "Act II, Scene 1"}]
}

Pieces are the document's own divisions: chapters, acts and scenes, sections, dated entries, poems, stories. A paper's pieces are its sections, the abstract first. Aim for pieces under %d words; where a division is longer, use its next level down. A part with no piece beginning in it gets an empty pieces list.

TEXT:
%s
"""

FIRST = ("source_class", "title", "author", "date", "body_start", "toc_count")   # first answer wins
LAST = ("end_matter_start",)                                                  # last answer wins


def slices(text):
    out, start = [], 0
    while start < len(text):
        end = min(len(text), start + SLICE)
        if end < len(text):
            cut = text.rfind("\n", start, end)
            end = cut + 1 if cut > start else end
        out.append(text[start:end])
        start = end
    return out


def propose(doc, model=MODEL):
    """One merged reply for the document, from the same question asked of every slice."""
    parts = slices(doc["text"])
    reply = {"pieces": []}
    for i, part in enumerate(parts):
        r = generate(PROMPT % (i + 1, len(parts), CAP_WORDS, part), model=model)
        for key in FIRST:
            if reply.get(key) is None and r.get(key) is not None:
                reply[key] = r[key]
        for key in LAST:
            if r.get(key) is not None:
                reply[key] = r[key]
        reply["pieces"] += r.get("pieces") or []
    return reply


In [ ]:
# Block 6: run the split over a spread of documents and find the character position of
# every break. Headings after the first are searched forward, each from the previous find.
# The first heading is the last occurrence before the second, so a contents entry identical
# to it (a bare "Introduction") is passed over. The body ends where the end matter starts,
# or at the end of the text when the model reports none. A quoted line that is not there is
# reported as missing.


def last_before(text, line, limit):
    at, found = locate(text, line, 0), None
    while at is not None and at < limit:
        found, at = at, locate(text, line, at + 1)
    return found


def breaks_for(doc, reply):
    text = doc["text"]
    breaks, missing = [], []
    pos = locate(text, reply.get("body_start"), 0) or 0
    for i, piece in enumerate(reply["pieces"]):
        if i == 0 and len(reply["pieces"]) > 1:
            continue                                   # placed once the second is known
        at = locate(text, piece.get("marker"), pos)
        if at is None:
            missing.append((piece.get("marker"), piece.get("title")))
            continue
        breaks.append((at, piece.get("title") or piece.get("marker")))
        pos = at + 1
    if len(reply["pieces"]) > 1:
        first = reply["pieces"][0]
        at = last_before(text, first.get("marker"), breaks[0][0]) if breaks else None
        if at is None:
            missing.append((first.get("marker"), first.get("title")))
        else:
            breaks.insert(0, (at, first.get("title") or first.get("marker")))
    if reply.get("end_matter_start"):
        end = locate(text, reply["end_matter_start"], pos)
        if end is None:
            missing.append((reply["end_matter_start"], "END MATTER START"))
    else:
        end = len(text)
    return breaks, end, missing


SAMPLES = Path("/kaggle/working/samples")
SAMPLES.mkdir(parents=True, exist_ok=True)
(SAMPLES / "meeting-notes.md").write_text(
    "# Weekly sync, 2026-09-02\n\n## Attendees\n\nJustin, Fang\n\n## Decisions\n\n- Kaggle first, desktop later.\n"
    "- The extractor never reads a manifest.\n\n## Actions\n\n- Justin: unpack scripts by Friday.\n", encoding="utf-8")
(SAMPLES / "email.txt").write_text(
    "From: Xing Fang <fang@example.edu>\nTo: Justin Hoffman <justin@example.edu>\nDate: Wed, 3 Sep 2026 09:12:00 -0500\n"
    "Subject: Re: notebook\n\nJustin,\n\nThe chapter notebook looks good. Send the repository link when it is public.\n\nXing\n",
    encoding="utf-8")

TRIAL = [
    RAW / "oz" / "01_55.txt",                       # novel with a contents list
    RAW / "oz" / "02_54.txt",                       # novel with no chapter word, bare title lines
    RAW / "holmes" / "03_1661.txt",                 # story collection
    RAW / "greek" / "18_10523.txt",                 # verse play
    RAW / "greek" / "03_348.txt",                   # anthology of many works
    RAW / "greek" / "27_library00apolgoog.txt",     # Google scan, no Gutenberg markers
    RAW / "greek" / "30_heroidesamores00ovid.txt",  # Loeb bilingual OCR
    RAW / "graphrag-bench" / "Novel-30752.txt",     # one line, no line breaks at all
    RAW / "graphrag-bench" / "Novel-25646.txt",     # one line, CHAPTER tokens inside
    PAPERS / "edge2024-graphrag.pdf",               # paper
    SAMPLES / "meeting-notes.md",
    SAMPLES / "email.txt",
]

replies = {}
for path in TRIAL:
    doc = to_text(path)
    before = spend()
    reply = propose(doc)
    replies[path.name] = (doc, reply)
    breaks, end, missing = breaks_for(doc, reply)
    date = (reply.get("date") or {}).get("iso")
    print(f"{path.name:<28} {reply.get('source_class') or '-':<9} pieces {len(reply['pieces']):>3}  found {len(breaks):>3}"
          f"  missing {len(missing):>2}  toc {str(reply.get('toc_count')):>4}  body_end {end if end is not None else '-':>8}/{len(doc['text']):<8}"
          f"  date {date or '-':<10} author {'y' if reply.get('author') else '-'}  ${spend() - before:.3f}")
    for line, label in missing[:3]:
        print(f"      MISSING {label}: {line!r}")
print(f"\n${spend():.3f} spent")
